# Experiment 4c: LSTM-GATv2 Constrained by Rolling Pearson

GATv2 attention **masked by rolling Pearson correlation graph**.

- Pearson determines **which edges exist** (graph structure)
- GATv2 learns **the weight of each edge** (from LSTM hidden states)
- Like rolling GCN but with learned, asymmetric, data-dependent edge weights
- No residual connection

**Key question:** If we give GAT the "right" structure (from Pearson), can it learn better edge weights than raw correlation values?

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from empyrical import sharpe_ratio, sortino_ratio, max_drawdown, annual_return, annual_volatility, calmar_ratio
import random, tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
EXPERIMENT_NAME = "4c_GATv2_rolling_pearson"
SEED = 40
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023
VOL_TARGET = 0.15

# Rolling Pearson Configuration
CORRELATION_LOOKBACK = 20
CORRELATION_THRESHOLD = 0.5
USE_EQUITY_RETURNS = True  # Use equity log returns for correlation (same as rolling GCN)

# Model Configuration
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

# GATv2 Hyperparameters
HIDDEN_LAYER_SIZE = 32
GAT_UNITS = 16
ATTN_HEADS = 4
LSTM_DROPOUT = 0.3
ATTN_DROPOUT = 0.1
LEARNING_RATE = 0.001
MAX_GRADIENT_NORM = 1.0
BATCH_SIZE = 64

if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
else:
    RESULTS_BASE = "FINAL_RESULTS"

returns_type = "equity" if USE_EQUITY_RETURNS else "straddle"
CONFIG_NAME = f"lb{CORRELATION_LOOKBACK}_th{CORRELATION_THRESHOLD}_{returns_type}"

print(f"Experiment: {EXPERIMENT_NAME} (seed={SEED})")
print(f"Config: {CONFIG_NAME}")
print(f"  Correlation lookback: {CORRELATION_LOOKBACK}, threshold: {CORRELATION_THRESHOLD}")
print(f"  Returns type: {returns_type}")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}, GAT: {GAT_UNITS}x{ATTN_HEADS}")
print(f"  Residual: REMOVED")

## 3. Helper Functions

In [ ]:
def calc_daily_returns(df, returns_col="captured_returns"):
    num_tickers = df["identifier"].nunique()
    daily_ret = df.groupby("time")[returns_col].sum() / num_tickers
    daily_ret.index = pd.to_datetime(daily_ret.index)
    return daily_ret.sort_index()

def calc_vol_scaled_returns(daily_returns, target_vol=0.15):
    current_vol = daily_returns.std() * np.sqrt(252)
    return daily_returns * (target_vol / current_vol) if current_vol > 0 else daily_returns

def calc_metrics(daily_returns, name="Strategy"):
    return {"Strategy": name, "E[Ret.]": annual_return(daily_returns),
        "Vol.": annual_volatility(daily_returns), "Sharpe": sharpe_ratio(daily_returns),
        "Sortino": sortino_ratio(daily_returns), "Max DD": -max_drawdown(daily_returns),
        "Calmar": calmar_ratio(daily_returns), "Hit Rate": (daily_returns > 0).mean(),
        "Avg P/L": daily_returns[daily_returns > 0].mean() / abs(daily_returns[daily_returns < 0].mean()) if (daily_returns < 0).any() else np.nan}

def calc_metrics_vol_normalized(daily_returns, name, target_vol=0.15):
    scaled = calc_vol_scaled_returns(daily_returns, target_vol)
    return calc_metrics(scaled, name + " (Vol-Norm)"), scaled

def display_metrics(m):
    df = pd.DataFrame([m]).set_index("Strategy")
    for c in ["E[Ret.]","Vol.","Max DD","Hit Rate"]: 
        if c in df.columns: df[c] = df[c].apply(lambda x: f"{x:.2%}")
    for c in ["Sharpe","Sortino","Calmar","Avg P/L"]: 
        if c in df.columns: df[c] = df[c].apply(lambda x: f"{x:.3f}")
    display(df)

def calc_yearly_sharpes(daily_returns):
    return {y: sharpe_ratio(daily_returns[daily_returns.index.year == y]) for y in sorted(daily_returns.index.year.unique())}

def plot_results(daily_returns_dict, title):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = plt.cm.tab10(np.linspace(0, 1, len(daily_returns_dict)))
    for (n, r), c in zip(daily_returns_dict.items(), colors):
        axes[0,0].plot(((1+r).cumprod()-1).index, ((1+r).cumprod()-1).values, label=n, lw=1.5, color=c)
        cum = (1+r).cumprod(); dd = (cum - cum.cummax()) / cum.cummax()
        axes[0,1].fill_between(dd.index, dd.values, 0, alpha=0.3, label=n, color=c)
        rs = r.rolling(252).mean() / r.rolling(252).std() * np.sqrt(252)
        axes[1,0].plot(rs.index, rs.values, label=n, lw=1, color=c)
    axes[0,0].set_title("Cumulative Returns"); axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)
    axes[0,1].set_title("Drawdown"); axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)
    axes[1,0].axhline(y=0, color="black", ls="--", lw=0.5); axes[1,0].set_title("Rolling Sharpe"); axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)
    yd = pd.DataFrame({n: calc_yearly_sharpes(r) for n,r in daily_returns_dict.items()})
    yd.plot(kind="bar", ax=axes[1,1], width=0.8); axes[1,1].axhline(y=0, color="black", ls="--", lw=0.5)
    axes[1,1].set_title("Yearly Sharpe"); axes[1,1].tick_params(axis="x", rotation=45); axes[1,1].grid(True, alpha=0.3, axis="y")
    plt.suptitle(title, fontsize=14, fontweight="bold"); plt.tight_layout(); plt.show()

## 4. Data Loading (Rolling Pearson)

In [ ]:
features_path = "data/straddle_features/features.csv"
if 'google.colab' in str(get_ipython()):
    features_path = "/content/drive/MyDrive/features.csv"

df = pd.read_csv(features_path)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df)} rows, {df['ticker'].nunique()} tickers")

In [ ]:
from gml.graph_model_inputs import RollingGraphModelFeatures

rolling_features = RollingGraphModelFeatures(
    df=df,
    total_time_steps=TOTAL_TIME_STEPS,
    correlation_lookback=CORRELATION_LOOKBACK,
    correlation_threshold=CORRELATION_THRESHOLD,
    returns_column="daily_returns",
    use_equity_returns=USE_EQUITY_RETURNS,
    start_boundary=TRAIN_START,
    test_boundary=TEST_START,
    test_end=TEST_END,
    train_valid_ratio=TRAIN_VALID_RATIO,
    split_tickers_individually=True,
    time_features=False,
)

datasets = rolling_features.make_rolling_graph_dataset(sliding_window=True)
train_data = datasets["train"]
valid_data = datasets["valid"]
test_data = datasets["test"]

print(f"
Training: inputs={train_data['inputs'].shape}, adj={train_data['adjacency'].shape}")
print(f"Validation: inputs={valid_data['inputs'].shape}, adj={valid_data['adjacency'].shape}")
print(f"Test: inputs={test_data['inputs'].shape}, adj={test_data['adjacency'].shape}")

## 5. Model Definition

In [ ]:
from gml.graph_attention_v2 import build_lstm_gat_rolling

num_tickers = train_data["inputs"].shape[1]
time_steps = train_data["inputs"].shape[2]
input_size = train_data["inputs"].shape[3]

print(f"Building LSTM-GATv2 Rolling Pearson (4c):")
print(f"  num_tickers: {num_tickers}, time_steps: {time_steps}, input_size: {input_size}")

model = build_lstm_gat_rolling(
    num_tickers=num_tickers, time_steps=time_steps, input_size=input_size,
    hidden_layer_size=HIDDEN_LAYER_SIZE, gat_units=GAT_UNITS, attn_heads=ATTN_HEADS,
    lstm_dropout=LSTM_DROPOUT, attn_dropout=ATTN_DROPOUT,
    learning_rate=LEARNING_RATE, max_gradient_norm=MAX_GRADIENT_NORM,
)

print(f"
Total parameters: {model.count_params():,}")

## 6. Training

In [ ]:
X_train = [train_data["inputs"], train_data["adjacency"]]
y_train = train_data["outputs"]
w_train = train_data["active_entries"]

X_valid = [valid_data["inputs"], valid_data["adjacency"]]
y_valid = valid_data["outputs"]
w_valid = valid_data["active_entries"]

print(f"Training: {train_data['inputs'].shape[0]} samples")
print(f"Validation: {valid_data['inputs'].shape[0]} samples")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1)

print("=" * 60)
print(f"Training {EXPERIMENT_NAME} ({CONFIG_NAME}, seed={SEED})")
print(f"  Pearson mask: lookback={CORRELATION_LOOKBACK}, threshold={CORRELATION_THRESHOLD}")
print(f"  NO RESIDUAL")
print("=" * 60)

history = model.fit(
    X_train, y_train, sample_weight=w_train,
    validation_data=(X_valid, y_valid, w_valid),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stopping], verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.xlabel("Epoch"); plt.ylabel("Loss (Neg Sharpe)")
plt.title(f"{EXPERIMENT_NAME}: Training Loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f"Epochs: {len(history.history['loss'])}, Best val: {min(history.history['val_loss']):.4f}")

## 7. Evaluation

In [ ]:
X_test = [test_data["inputs"], test_data["adjacency"]]
predictions = model.predict(X_test)
print(f"Predictions shape: {predictions.shape}")

In [ ]:
positions = predictions[:, :, -1, 0].reshape(-1)
returns = test_data["outputs"][:, :, -1, 0].reshape(-1)
captured_returns = positions * returns
dates = test_data["date"][:, :, -1, 0].reshape(-1)
identifiers = test_data["identifier"][:, :, -1, 0].reshape(-1)

results_df = pd.DataFrame({
    "time": dates, "identifier": identifiers,
    "position": positions, "returns": returns, "captured_returns": captured_returns,
})
results_df["time"] = pd.to_datetime(results_df["time"])
results_df = results_df[results_df["identifier"] != "0"]
print(f"Results: {len(results_df)} rows")

In [ ]:
daily_returns = calc_daily_returns(results_df)

print("
" + "=" * 60)
print(f"{EXPERIMENT_NAME} ({CONFIG_NAME}) Results")
print("=" * 60)
metrics_raw = calc_metrics(daily_returns, f"{EXPERIMENT_NAME}")
display_metrics(metrics_raw)

print(f"
Vol-Normalized (Target: {VOL_TARGET:.0%})")
metrics_norm, _ = calc_metrics_vol_normalized(daily_returns, EXPERIMENT_NAME, VOL_TARGET)
display_metrics(metrics_norm)

print("
Yearly Sharpes:")
yearly_sharpes = calc_yearly_sharpes(daily_returns)
for y, s in yearly_sharpes.items(): print(f"  {y}: {s:.4f}")

In [ ]:
plot_results({EXPERIMENT_NAME: daily_returns}, f"{EXPERIMENT_NAME} ({TEST_START}-{TEST_END})")

## 8. Attention Analysis

In [ ]:
from gml.experiment_utils import compute_graph_stats

# Extract attention weights from the masked GAT layer
gat_layer = model.get_layer("dynamic_masked_gat")
W_src = gat_layer.W_src.numpy()
W_dst = gat_layer.W_dst.numpy()
a = gat_layer.a.numpy()
num_heads = W_src.shape[0]

# Build sub-model to get LSTM hidden states
gat_input_tensor = gat_layer.input[0]  # features input to GAT
sub_model = tf.keras.Model(inputs=model.input, outputs=gat_input_tensor)

# Get LSTM outputs for test set
print(f"Extracting LSTM hidden states for {test_data['inputs'].shape[0]} test windows...")
lstm_outputs = sub_model.predict(X_test, verbose=0)
print(f"LSTM outputs shape: {lstm_outputs.shape}")

# Compute masked attention for each test window
# Use the last timestep's hidden states for attention computation
test_adj = test_data["adjacency"]
all_attn = []

for idx in range(lstm_outputs.shape[0]):
    h = lstm_outputs[idx, -1, :, :]  # (nodes, hidden) — last timestep
    adj = test_adj[idx]  # (nodes, nodes)
    mask = ((adj + np.eye(adj.shape[0])) > 0).astype(np.float32)

    head_attns = []
    for head_idx in range(num_heads):
        h_s = h @ W_src[head_idx]
        h_d = h @ W_dst[head_idx]
        pw = h_s[:, np.newaxis, :] + h_d[np.newaxis, :, :]
        pw = np.where(pw > 0, pw, 0.2 * pw)
        scores = (pw @ a[head_idx]).squeeze(-1)
        scores = np.where(mask > 0, scores, -1e9)
        exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
        attn = exp_s / (exp_s.sum(axis=-1, keepdims=True) + 1e-9)
        head_attns.append(attn)
    all_attn.append(np.stack(head_attns, axis=0))

all_attn = np.array(all_attn)  # (N, heads, nodes, nodes)
all_graphs_avg = all_attn.mean(axis=1)  # (N, nodes, nodes)
print(f"Attention shape: {all_attn.shape}")

test_dates_arr = pd.to_datetime(test_data["date"][:, 0, -1, 0])

# Entropy (only over non-masked neighbors)
entropy = -np.sum(all_graphs_avg * np.log(all_graphs_avg + 1e-9), axis=-1)
mean_entropy = entropy.mean()
max_entropy = np.log(all_graphs_avg.shape[-1])
print(f"
Attention entropy: {mean_entropy:.3f} (uniform over all nodes = {max_entropy:.3f})")

# For masked attention, uniform over ~k neighbors has entropy log(k)
avg_degree = (test_adj > 0).sum(axis=-1).mean()
expected_masked_uniform = np.log(avg_degree + 1)  # +1 for self-loop
print(f"Expected uniform over avg neighborhood ({avg_degree:.0f} neighbors): {expected_masked_uniform:.3f}")
print(f"Entropy ratio vs neighborhood: {mean_entropy / expected_masked_uniform:.3f}")

EDGE_THRESHOLD = 0.0  # All edges in mask are valid
graph_stats = compute_graph_stats(all_graphs_avg, threshold=EDGE_THRESHOLD)

In [ ]:
# Heatmap: compare Pearson adjacency vs learned attention
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Average Pearson adjacency
avg_pearson = test_adj.mean(axis=0)
sns.heatmap(avg_pearson, ax=axes[0], cmap="viridis", vmin=0)
axes[0].set_title("Average Pearson Adjacency (test)")

# Average learned attention
avg_attn = all_graphs_avg.mean(axis=0)
sns.heatmap(avg_attn, ax=axes[1], cmap="viridis", vmin=0)
axes[1].set_title("Average Learned Attention (test)")

# Difference
diff = avg_attn - avg_pearson
sns.heatmap(diff, ax=axes[2], cmap="RdBu_r", center=0)
axes[2].set_title("Difference (Learned - Pearson)")

plt.suptitle(f"{EXPERIMENT_NAME}: Pearson vs Learned Attention", fontweight="bold")
plt.tight_layout(); plt.show()

## 9. Graph Statistics Over Time

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
uniform_ent = np.log(all_graphs_avg.shape[-1])

axes[0,0].plot(test_dates_arr, graph_stats["num_edges"], lw=0.8); axes[0,0].set_ylabel("Edges"); axes[0,0].set_title("Edge Count"); axes[0,0].grid(True, alpha=0.3)
axes[0,1].plot(test_dates_arr, graph_stats["mean_degree"], lw=0.8, label="Mean"); axes[0,1].fill_between(test_dates_arr, graph_stats["min_degree"], graph_stats["max_degree"], alpha=0.2); axes[0,1].set_ylabel("Degree"); axes[0,1].set_title("Node Degree"); axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)
axes[1,0].plot(test_dates_arr, graph_stats["mean_edge_weight"], lw=0.8); axes[1,0].set_ylabel("Mean Weight"); axes[1,0].set_title("Mean Attention Weight"); axes[1,0].grid(True, alpha=0.3)
axes[1,1].plot(test_dates_arr, graph_stats["std_edge_weight"], lw=0.8); axes[1,1].set_ylabel("Std Weight"); axes[1,1].set_title("Std of Weights"); axes[1,1].grid(True, alpha=0.3)
axes[2,0].plot(test_dates_arr, graph_stats["max_attn"], lw=0.8); axes[2,0].set_ylabel("Max Attn"); axes[2,0].set_xlabel("Date"); axes[2,0].set_title("Max Pairwise Attention"); axes[2,0].grid(True, alpha=0.3)
axes[2,1].plot(test_dates_arr, graph_stats["mean_entropy"], lw=0.8); axes[2,1].axhline(y=uniform_ent, color="red", ls="--", lw=0.8, label=f"Uniform(all)={uniform_ent:.2f}"); axes[2,1].axhline(y=expected_masked_uniform, color="orange", ls="--", lw=0.8, label=f"Uniform(nbrs)={expected_masked_uniform:.2f}"); axes[2,1].set_ylabel("Entropy"); axes[2,1].set_xlabel("Date"); axes[2,1].set_title("Mean Entropy"); axes[2,1].legend(fontsize=8); axes[2,1].grid(True, alpha=0.3)
plt.suptitle(f"{EXPERIMENT_NAME}: Graph Stats Over Time", fontweight="bold"); plt.tight_layout(); plt.show()

## 10. Interactive Graph Visualization

In [ ]:
import networkx as nx
import ipywidgets as widgets
from IPython.display import display, clear_output
from settings.default import ALL_TICKERS, BBG_SECTORS

SECTOR_COLORS = {
    "Information Technology": "#1f77b4", "Healthcare": "#2ca02c",
    "Financials": "#ff7f0e", "Consumer Discretionary": "#d62728",
    "Consumer Staples": "#9467bd", "Industrials": "#8c564b",
    "Communication Services": "#e377c2", "Energy": "#7f7f7f",
    "Utilities": "#bcbd22", "Real Estate": "#17becf",
}
tickers = sorted(ALL_TICKERS)
G_ref = nx.Graph()
for t in tickers: G_ref.add_node(t)
fixed_pos = nx.spring_layout(G_ref, k=2.5, iterations=100, seed=42)
node_colors = [SECTOR_COLORS.get(BBG_SECTORS.get(t, "Unknown"), "#cccccc") for t in tickers]
rolling_sharpe = daily_returns.rolling(252).mean() / daily_returns.rolling(252).std() * np.sqrt(252)

VIZ_THRESHOLD = 0.05  # Higher threshold since attention is sparse

output_widget = widgets.Output()
def update_graph(window_idx):
    with output_widget:
        clear_output(wait=True)
        fig, (ax_g, ax_s) = plt.subplots(2, 1, figsize=(14, 16), gridspec_kw={"height_ratios": [3, 1]})
        adj = all_graphs_avg[window_idx]
        date_str = str(test_dates_arr[window_idx].date()) if window_idx < len(test_dates_arr) else "N/A"
        G = nx.Graph()
        for t in tickers: G.add_node(t)
        n = len(tickers)
        for i in range(n):
            for j in range(i+1, n):
                w = (adj[i,j] + adj[j,i]) / 2
                if w > VIZ_THRESHOLD: G.add_edge(tickers[i], tickers[j], weight=w)
        edges = G.edges(data=True)
        if len(edges) > 0:
            max_w = max(d["weight"] for _,_,d in edges)
            for (u,v,d) in edges:
                w = d["weight"]; ax_g.plot([fixed_pos[u][0], fixed_pos[v][0]], [fixed_pos[u][1], fixed_pos[v][1]], color="gray", lw=2*w/max_w, alpha=0.3+0.7*w/max_w, zorder=1)
        nx.draw_networkx_nodes(G, fixed_pos, node_color=node_colors, node_size=600, alpha=0.9, ax=ax_g)
        nx.draw_networkx_labels(G, fixed_pos, font_size=6, font_weight="bold", ax=ax_g)
        ax_g.set_title(f"Learned Attention at {date_str} | {G.number_of_edges()} edges (th={VIZ_THRESHOLD})", fontsize=14, fontweight="bold"); ax_g.axis("off")
        ax_s.plot(rolling_sharpe.index, rolling_sharpe.values, lw=1, color="blue")
        ax_s.axhline(y=0, color="black", ls="--", lw=0.5)
        if window_idx < len(test_dates_arr): ax_s.axvline(x=test_dates_arr[window_idx], color="red", lw=2, alpha=0.8)
        ax_s.set_title("Rolling 252-Day Sharpe"); ax_s.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

slider = widgets.IntSlider(min=0, max=len(all_graphs_avg)-1, step=1, value=0, description="Window:", continuous_update=False, layout=widgets.Layout(width="80%"))
widgets.interactive(update_graph, window_idx=slider)
display(slider, output_widget)
update_graph(0)

## 11. Position Analysis

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(results_df["position"], bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Position"); plt.ylabel("Frequency")
plt.title(f"{EXPERIMENT_NAME}: Position Distribution")
plt.axvline(x=0, color="red", ls="--", lw=1); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 12. Save Results

In [ ]:
from gml.experiment_utils import save_experiment_results

hyperparams = {
    "hidden_layer_size": HIDDEN_LAYER_SIZE, "gat_units": GAT_UNITS,
    "attn_heads": ATTN_HEADS, "lstm_dropout": LSTM_DROPOUT,
    "attn_dropout": ATTN_DROPOUT, "learning_rate": LEARNING_RATE,
    "max_gradient_norm": MAX_GRADIENT_NORM, "batch_size": BATCH_SIZE,
    "residual": False,
    "model_type": "GATv2_rolling_pearson_masked",
    "correlation_lookback": CORRELATION_LOOKBACK,
    "correlation_threshold": CORRELATION_THRESHOLD,
    "use_equity_returns": USE_EQUITY_RETURNS,
    "total_time_steps": TOTAL_TIME_STEPS,
    "train_start": TRAIN_START, "test_start": TEST_START, "test_end": TEST_END,
}

save_experiment_results(
    experiment_name=EXPERIMENT_NAME, seed=SEED, config_name=CONFIG_NAME,
    predictions=predictions, results_df=results_df,
    daily_returns=daily_returns, metrics_raw=metrics_raw,
    metrics_norm=metrics_norm, yearly_sharpes=yearly_sharpes,
    training_history=history.history, hyperparams=hyperparams,
    test_dates=test_dates_arr.values,
    attention_weights=all_attn,
    adjacency=test_data["adjacency"],
    graph_stats=graph_stats,
    model=model,
    base_dir=RESULTS_BASE,
)

## 13. Summary

In [ ]:
print("=" * 60)
print(f"EXPERIMENT: {EXPERIMENT_NAME} ({CONFIG_NAME}, seed={SEED})")
print("=" * 60)
print(f"Model: LSTM-GATv2 constrained by rolling Pearson (NO RESIDUAL)")
print(f"Pearson: lookback={CORRELATION_LOOKBACK}, threshold={CORRELATION_THRESHOLD}, {returns_type}")
print(f"Sharpe: {metrics_raw['Sharpe']:.3f}, Return: {metrics_raw['E[Ret.]']:.2%}, Vol: {metrics_raw['Vol.']:.2%}")
print(f"Entropy: {mean_entropy:.3f} (uniform over nbrs: {expected_masked_uniform:.3f})")
print(f"Saved to: {RESULTS_BASE}/{EXPERIMENT_NAME}/{CONFIG_NAME}/seed_{SEED}/")